In [20]:
import pandas as pd 
import plotly.express as px
import matplotlib.pyplot as plt 
import seaborn as sns   
import numpy as np 

In [21]:
PATH = '/Users/Marcy_Student/Desktop/Capstone/nyc-dog-licensing_expanded/data/Cleaned /final_dataset.csv'
df = pd.read_csv(PATH)

In [22]:
# Add new dataset with projected population by borough from 2020 - 2030 
pop_df = pd.read_csv("/Users/Marcy_Student/Desktop/Capstone/nyc-dog-licensing_expanded/data/Cleaned /New_York_City_Population_by_Borough,_1950_-_2040_20260407.csv")

# Keep only relevant columns
pop_df = pop_df[['Borough', '2020']]

# Rename for clarity
pop_df = pop_df.rename(columns={'2020': 'Population'})

# Remove NYC Total row
pop_df = pop_df[pop_df['Borough'] != 'NYC Total']

# Convert population to numeric
pop_df['Population'] = pop_df['Population'].str.replace(',', '').astype(int)

# Standardize borough names
pop_df['Borough'] = pop_df['Borough'].str.strip().str.title()

pop_df


,Borough,Population
1,Bronx,1446788
2,Brooklyn,2648452
3,Manhattan,1638281
4,Queens,2330295
5,Staten Island,487155


In [23]:
df['Borough'] = df['Borough'].str.strip().str.title()

# Drop null boroughs
df = df.dropna(subset=['Borough'])

# Convert IssueMonth to numeric
df['IssueMonth'] = pd.to_datetime(df['IssueMonth'], format='%B').dt.month

# Create datetime column
df['IssueDate'] = pd.to_datetime(
    df['IssueYear'].astype(str) + '-' + df['IssueMonth'].astype(str)
)

df.head()

,Name,Gender,BirthYear,Breed,ZipCode,IssuedDate,ExpiredDate,ExtractYear,Borough,DogAgeAtLicense,IssueYear,IssueMonth,IssueDate
0,PAIGE,F,2014.0,American Pit Bull Mix / Pit Bull Mix,10035.0,2014-09-12,2017-09-12,2016,Manhattan,0.0,2014,9,2014-09-01
1,YOGI,M,2010.0,Boxer,10465.0,2014-09-12,2017-10-02,2016,The Bronx,4.0,2014,9,2014-09-01
2,ALI,M,2014.0,Basenji,10013.0,2014-09-12,2019-09-12,2016,Manhattan,0.0,2014,9,2014-09-01
3,QUEEN,F,2013.0,Akita Crossbreed,10013.0,2014-09-12,2017-09-12,2016,Manhattan,1.0,2014,9,2014-09-01
4,LOLA,F,2009.0,Maltese,10028.0,2014-09-12,2017-10-09,2016,Manhattan,5.0,2014,9,2014-09-01


In [24]:
# Fix borough name from the bronx to simply bronx for both datsets

def clean_borough(x):
    return (
        str(x)
        .strip()
        .replace("The Bronx", "Bronx")
        .title()
    )

df['Borough'] = df['Borough'].apply(clean_borough)
pop_df['Borough'] = pop_df['Borough'].apply(clean_borough)

In [25]:
# Monthly Dog Registrations
monthly_trend = (
    df.groupby(pd.Grouper(key='IssueDate', freq='M'))
      .size()
      .reset_index(name='dog_registrations')
)

monthly_trend.head()

/var/folders/tb/3_577n455q9bf55gxv6lrs4w0000gp/T/ipykernel_93143/2567381970.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key='IssueDate', freq='M'))


,IssueDate,dog_registrations
0,2014-09-30,236
1,2014-10-31,374
2,2014-11-30,264
3,2014-12-31,674
4,2015-01-31,2177


In [26]:
# Drop any unknown boroughs - about 2,504 
df = df[df['Borough'] != 'Unknown']
df = df.dropna(subset=['Borough'])

dogs_per_capita = (
    df.groupby('Borough')
      .size()
      .reset_index(name='dog_count')
      .merge(pop_df, on='Borough', how='left')
)

dogs_per_capita['dogs_per_1000'] = (
    dogs_per_capita['dog_count'] / dogs_per_capita['Population'] * 1000
)

dogs_per_capita.sort_values('dogs_per_1000', ascending=False)

,Borough,dog_count,Population,dogs_per_1000
2,Manhattan,241353,1638281,147.320881
4,Staten Island,59735,487155,122.620111
1,Brooklyn,194153,2648452,73.308106
3,Queens,148539,2330295,63.742573
0,Bronx,74519,1446788,51.506510


In [27]:
borough_summary = (
    df.groupby('Borough')
      .agg(
          dog_count=('Name', 'count'),
          avg_dog_age=('DogAgeAtLicense', 'mean'),
          unique_breeds=('Breed', 'nunique')
      )
      .reset_index()
)

In [28]:
borough_summary = borough_summary.merge(pop_df, on='Borough', how='left')

In [29]:
# Dogs per 1,000 residents
borough_summary['dogs_per_1000'] = (
    borough_summary['dog_count'] / borough_summary['Population'] * 1000
)

# % of total dogs
total_dogs = borough_summary['dog_count'].sum()

borough_summary['pct_of_total_dogs'] = (
    borough_summary['dog_count'] / total_dogs * 100
)

# Rank boroughs
borough_summary['density_rank'] = (
    borough_summary['dogs_per_1000']
    .rank(ascending=False)
)

In [30]:
# segmentation
def density_segment(x):
    if x > 100:
        return "High Density"
    elif x > 50:
        return "Medium Density"
    else:
        return "Low Density"

borough_summary['density_segment'] = borough_summary['dogs_per_1000'].apply(density_segment)

In [31]:
# clean column names 
borough_summary = borough_summary[[
    'Borough',
    'dog_count',
    'Population',
    'dogs_per_1000',
    'pct_of_total_dogs',
    'unique_breeds',
    'avg_dog_age',
    'density_rank',
    'density_segment'
]]

In [32]:
# exports summary data to csv for tableu
borough_summary.to_csv("borough_summary_tableau.csv", index=False)

In [33]:
# fact table for regristations

fact_table = df[[
    'Breed',
    'Borough',
    'IssueDate',
    'IssueYear',
    'IssueMonth',
    'DogAgeAtLicense'
]].copy()

fact_table['Year'] = fact_table['IssueDate'].dt.year
fact_table['Month'] = fact_table['IssueDate'].dt.month
fact_table['MonthName'] = fact_table['IssueDate'].dt.strftime('%B')


fact_table.to_csv("fact_dog_licenses.csv", index=False)


